# Vertex-wise Ridge Encoding for Workbench Maps

Fixed event table version: one row is one audio/model chunk, not one video.

Settings: delay = 5 s, model normalization = demean only, fMRI normalization = none.


In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import scipy.io as sio
import nibabel as nib

from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# 2. USER CONFIG
# ============================================================

HEMI = "RIGHT"   # "RIGHT" or "LEFT"

BASE_DIR = r"G:\Rajimehr\Our Project Results"

# ---------- fMRI dense surface file ----------
# This must be real vertex-wise data, timepoints x vertices.
# If this file is parcel-expanded, the output will still look parcel/blocky.
if HEMI.upper() == "RIGHT":
    FMRIMAT_FILE = os.path.join(BASE_DIR, "HCP Data", "notmean_right_Meanfile.mat")
    FMRI_VAR_NAME = "Right_data"
elif HEMI.upper() == "LEFT":
    FMRIMAT_FILE = os.path.join(BASE_DIR, "HCP Data", "notmean_left_Meanfile.mat")
    FMRI_VAR_NAME = "Left_data"
else:
    raise ValueError("HEMI must be RIGHT or LEFT")

# ---------- Timing and labels ----------
TOI_FILE = os.path.join(BASE_DIR, "TOI_V3.xlsx")
AUDIO_LABELS_FILE = os.path.join(BASE_DIR, "Audio_Labels.xlsx")

# ---------- Model features ----------
MODEL_NAME = "PEAV"
MODEL_FEATURE_FILE = os.path.join(BASE_DIR, "Amin's outputs", "pe-av-small-16-frame.npz")
MODEL_VAR_NAME = "av"

# ---------- Output ----------
OUT_DIR = os.path.join(BASE_DIR, f"VERTEXWISE_RIDGE_dmean_delay5_noFMRIz_{MODEL_NAME}_{HEMI}")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Timing ----------
TR = 1.0
DELAY_SEC = 5.0
WINDOW_SEC = 3.0

# ---------- Ridge ----------
RIDGE_ALPHA = 100.0
N_SPLITS = 5
RANDOM_SEED = 42

# Model normalization: demean only
MODEL_NORMALIZATION = "demean_only"

# fMRI normalization: none
FMRI_NORMALIZATION = "none"

CONDITIONS_TO_RUN = [
    "all",
    "speech_only",
    "environment_only",
    "music_only",
]

print("FMRIMAT_FILE:", FMRIMAT_FILE)
print("FMRI_VAR_NAME:", FMRI_VAR_NAME)
print("MODEL_FEATURE_FILE:", MODEL_FEATURE_FILE)
print("TOI_FILE:", TOI_FILE)
print("AUDIO_LABELS_FILE:", AUDIO_LABELS_FILE)
print("OUT_DIR:", OUT_DIR)
print("DELAY_SEC:", DELAY_SEC)
print("MODEL_NORMALIZATION:", MODEL_NORMALIZATION)
print("FMRI_NORMALIZATION:", FMRI_NORMALIZATION)

FMRIMAT_FILE: G:\Rajimehr\Our Project Results\HCP Data\notmean_right_Meanfile.mat
FMRI_VAR_NAME: Right_data
MODEL_FEATURE_FILE: G:\Rajimehr\Our Project Results\Amin's outputs\pe-av-small-16-frame.npz
TOI_FILE: G:\Rajimehr\Our Project Results\TOI_V3.xlsx
AUDIO_LABELS_FILE: G:\Rajimehr\Our Project Results\Audio_Labels.xlsx
OUT_DIR: G:\Rajimehr\Our Project Results\VERTEXWISE_RIDGE_dmean_delay5_noFMRIz_PEAV_RIGHT
DELAY_SEC: 5.0
MODEL_NORMALIZATION: demean_only
FMRI_NORMALIZATION: none


In [3]:
# ============================================================
# 3. General loading helpers
# ============================================================

def _mat_keys(mat):
    return [k for k in mat.keys() if not k.startswith("__")]


def load_mat_2d_array(mat_file, var_name=None, preferred_names=None):
    mat = sio.loadmat(mat_file, squeeze_me=True, struct_as_record=False)
    keys = _mat_keys(mat)
    print("MAT keys:", keys)

    if var_name is not None:
        if var_name not in mat:
            raise KeyError(f"{var_name} not found. Available keys: {keys}")
        arr = np.asarray(mat[var_name])
        return arr, var_name

    if preferred_names is None:
        preferred_names = []

    for name in preferred_names:
        if name in mat:
            arr = np.asarray(mat[name])
            if arr.ndim == 2:
                return arr, name

    candidates = []
    for k in keys:
        arr = np.asarray(mat[k])
        if arr.ndim == 2 and np.issubdtype(arr.dtype, np.number):
            candidates.append((k, arr.shape))

    if not candidates:
        raise ValueError("Could not find any 2D numeric array in mat file.")

    print("2D candidates:", candidates)
    chosen = candidates[0][0]
    return np.asarray(mat[chosen]), chosen


def ensure_timepoints_by_vertices(data):
    data = np.asarray(data, dtype=np.float32)
    if data.ndim != 2:
        raise ValueError(f"Expected 2D data, got shape {data.shape}")

    # HCP fsLR hemisphere usually has 32492 vertices.
    # If data is vertices x timepoints, transpose it.
    if data.shape[0] in [32492, 59412, 91282] and data.shape[1] < data.shape[0]:
        print("Detected vertices x timepoints. Transposing to timepoints x vertices.")
        data = data.T

    return data.astype(np.float32)


def load_model_features(model_file, model_var_name=None):
    ext = os.path.splitext(model_file)[1].lower()

    if ext == ".npz":
        f = np.load(model_file)
        print("NPZ keys:", f.files)
        if model_var_name is not None:
            key = model_var_name
        elif "av" in f.files:
            key = "av"
        elif "features" in f.files:
            key = "features"
        elif "model_features" in f.files:
            key = "model_features"
        else:
            key = f.files[0]
            print("Auto-selected key:", key)
        feats = np.asarray(f[key], dtype=np.float32)
        return feats, key

    if ext == ".mat":
        feats, name = load_mat_2d_array(
            model_file,
            var_name=model_var_name,
            preferred_names=["model_features", "features", "av", "video_features", "X", "embeddings"]
        )
        return np.asarray(feats, dtype=np.float32), name

    raise ValueError(f"Unsupported model feature file type: {ext}")

In [4]:
# ============================================================
# 4. Load fMRI vertex-wise data
# ============================================================

raw_data, loaded_fmri_name = load_mat_2d_array(FMRIMAT_FILE, var_name=FMRI_VAR_NAME)
data = ensure_timepoints_by_vertices(raw_data)

print("Loaded fMRI variable:", loaded_fmri_name)
print("data shape:", data.shape)
print("Expected: timepoints x vertices")

# No fMRI normalization in this notebook.
# Only replace non-finite values with 0 to avoid crashes.
# This is not z-scoring or scaling.
bad = ~np.isfinite(data)
print("Non-finite fMRI values:", int(bad.sum()))
if bad.any():
    data = data.copy()
    data[bad] = 0.0

print("Final data shape:", data.shape)

MAT keys: ['Right_data']
Detected vertices x timepoints. Transposing to timepoints x vertices.
Loaded fMRI variable: Right_data
data shape: (3655, 32492)
Expected: timepoints x vertices
Non-finite fMRI values: 0
Final data shape: (3655, 32492)


In [6]:
# ============================================================
# 5. Check if the fMRI is truly vertex-wise or parcel-expanded
# ============================================================

print("data shape:", data.shape)

for t in [0, 10, 50, 100]:
    if t < data.shape[0]:
        x = data[t, :]
        x = x[np.isfinite(x)]
        unique_count = len(np.unique(np.round(x, 6)))
        print("timepoint", t, "unique values:", unique_count, "out of", x.size)

print("Rule of thumb:")
print("- thousands of unique values = likely real vertex-wise")
print("- ~180 or ~360 unique values = likely parcellated/parcel-expanded")

data shape: (3655, 32492)
timepoint 0 unique values: 29251 out of 32492
timepoint 10 unique values: 29212 out of 32492
timepoint 50 unique values: 29092 out of 32492
timepoint 100 unique values: 29266 out of 32492
Rule of thumb:
- thousands of unique values = likely real vertex-wise
- ~180 or ~360 unique values = likely parcellated/parcel-expanded


In [7]:
# ============================================================
# 6. Build event_df correctly from TOI + Audio_Labels
#    One row = one audio/model chunk, NOT one video
# ============================================================

import re

toi = pd.read_excel(TOI_FILE)
audio = pd.read_excel(AUDIO_LABELS_FILE)

print("TOI columns:")
print(list(toi.columns))
print("\nAudio label columns:")
print(list(audio.columns))

display(toi.head())
display(audio.head())


def get_number(x):
    """Extract the first integer from strings like 'Audio_01', 'video12', etc."""
    nums = re.findall(r"\d+", str(x))
    if len(nums) == 0:
        raise ValueError(f"No number found in: {x}")
    return int(nums[0])


def find_first_existing_column(df, candidates):
    """Find a column using exact names first, then case-insensitive names."""
    cols = list(df.columns)
    lower_map = {str(c).lower().strip(): c for c in cols}

    for c in candidates:
        if c in df.columns:
            return c

    for c in candidates:
        key = str(c).lower().strip()
        if key in lower_map:
            return lower_map[key]

    return None


def label_to_binary(x):
    """Robust conversion for 0/1, yes/no, true/false labels."""
    if pd.isna(x):
        return 0

    s = str(x).strip().lower()

    if s in ["1", "1.0", "true", "yes", "y", "present"]:
        return 1

    if s in ["0", "0.0", "false", "no", "n", "absent", "none", "nan", ""]:
        return 0

    try:
        return int(float(s) > 0)
    except Exception:
        return 0


# ------------------------------------------------------------
# Standardize TOI columns
# TOI should describe the 18 videos/runs and their start times.
# ------------------------------------------------------------

toi = toi.copy()
audio = audio.copy()

toi_video_col = find_first_existing_column(
    toi,
    ["VideoNumber", "video", "Video", "Movie", "movie", "run", "Run", "AudioNumber"]
)

if toi_video_col is None:
    raise KeyError(f"Cannot find a video/run column in TOI. Columns: {list(toi.columns)}")

toi_start_col = find_first_existing_column(
    toi,
    ["Start_Stream", "start_stream", "Start", "start", "onset", "Onset", "onset_sec", "start_sec"]
)

if toi_start_col is None:
    raise KeyError(
        "Cannot find a start/onset column in TOI. "
        f"Expected something like Start_Stream. Columns: {list(toi.columns)}"
    )

toi["VideoNumber_clean"] = toi[toi_video_col].apply(get_number).astype(int)
toi["Start_Stream_clean"] = pd.to_numeric(toi[toi_start_col], errors="coerce")

if toi["Start_Stream_clean"].isna().any():
    bad_rows = toi[toi["Start_Stream_clean"].isna()]
    print("Rows with missing TOI start time:")
    display(bad_rows)
    raise ValueError("Some TOI start times could not be converted to numbers.")

print("\nDetected TOI video column:", toi_video_col)
print("Detected TOI start column:", toi_start_col)
print("Number of TOI rows/videos:", len(toi))


# ------------------------------------------------------------
# Standardize Audio_Labels columns
# Audio_Labels should contain one row per chunk/event.
# ------------------------------------------------------------

audio_video_col = find_first_existing_column(
    audio,
    ["AudioNumber", "VideoNumber", "video", "Video", "Movie", "movie", "run", "Run"]
)

if audio_video_col is None:
    raise KeyError(
        "Cannot find AudioNumber/VideoNumber in Audio_Labels. "
        f"Columns: {list(audio.columns)}"
    )

audio_chunk_col = find_first_existing_column(
    audio,
    ["ChunkNumber", "chunk", "Chunk", "event", "Event", "TR", "time"]
)

if audio_chunk_col is None:
    raise KeyError(
        "Cannot find ChunkNumber in Audio_Labels. "
        f"Columns: {list(audio.columns)}"
    )

speech_col = find_first_existing_column(audio, ["Speech", "speech"])
environment_col = find_first_existing_column(audio, ["Environment", "environment", "Env", "env"])
music_col = find_first_existing_column(audio, ["Music", "music"])

for name, col in [("Speech", speech_col), ("Environment", environment_col), ("Music", music_col)]:
    if col is None:
        raise KeyError(f"Missing {name} column in Audio_Labels. Columns: {list(audio.columns)}")

print("\nDetected Audio video column:", audio_video_col)
print("Detected Audio chunk column:", audio_chunk_col)
print("Detected label columns:", speech_col, environment_col, music_col)

audio["AudioNumber_clean"] = audio[audio_video_col].apply(get_number).astype(int)
audio["ChunkNumber_clean"] = audio[audio_chunk_col].apply(get_number).astype(int)
audio["Speech"] = audio[speech_col].apply(label_to_binary).astype(int)
audio["Environment"] = audio[environment_col].apply(label_to_binary).astype(int)
audio["Music"] = audio[music_col].apply(label_to_binary).astype(int)


# ------------------------------------------------------------
# Build one event per audio/model chunk
# ------------------------------------------------------------

# Your previous pipeline used 2-second audio/model chunks.
# Change this only if your PEAV features were extracted with a different chunk size.
BIN_SEC = 2.0

rows = []
global_event = 0

for _, video_row in toi.sort_values("VideoNumber_clean").iterrows():
    vid_num = int(video_row["VideoNumber_clean"])
    start_stream = float(video_row["Start_Stream_clean"])

    labels_vid = audio[audio["AudioNumber_clean"] == vid_num].copy()
    labels_vid = labels_vid.sort_values("ChunkNumber_clean").reset_index(drop=True)

    print(f"Video {vid_num}: {len(labels_vid)} chunks")

    for local_idx, lab in labels_vid.iterrows():
        onset_sec = start_stream + local_idx * BIN_SEC

        rows.append({
            "global_event": global_event,
            "VideoNumber": vid_num,
            "video": f"video{vid_num}",
            "local_event": local_idx,
            "ChunkNumber": int(lab["ChunkNumber_clean"]),
            "onset_sec": onset_sec,
            "duration_sec": BIN_SEC,
            "Speech": int(lab["Speech"]),
            "Environment": int(lab["Environment"]),
            "Music": int(lab["Music"]),
        })

        global_event += 1

event_df = pd.DataFrame(rows)

if len(event_df) == 0:
    raise ValueError(
        "event_df is empty. TOI video numbers probably do not match Audio_Labels AudioNumber values."
    )

print("\nevent_df created correctly.")
print("event_df shape:", event_df.shape)
display(event_df.head())
display(event_df.tail())

print("\nAudio label rows:", len(audio))
print("event_df rows:", len(event_df))

if len(event_df) != len(audio):
    print("WARNING: event_df length does not match Audio_Labels length.")
    print("This can happen if TOI is missing a video or if Audio_Labels contains extra videos.")
else:
    print("Good: event_df length matches Audio_Labels length.")

print("\nTiming check:")
print("First onset:", float(event_df["onset_sec"].iloc[0]))
print("Last offset:", float((event_df["onset_sec"] + event_df["duration_sec"]).iloc[-1]))
print("fMRI duration:", float(data.shape[0] * TR))


# ------------------------------------------------------------
# Count alone and mixed audio conditions
# ------------------------------------------------------------

speech = event_df["Speech"] == 1
environment = event_df["Environment"] == 1
music = event_df["Music"] == 1

print("\nTotal events:", len(event_df))

print("\nAlone conditions:")
print("Speech only:", int((speech & ~environment & ~music).sum()))
print("Environment only:", int((environment & ~speech & ~music).sum()))
print("Music only:", int((music & ~speech & ~environment).sum()))

print("\nMixed conditions:")
print("Speech + Environment only:", int((speech & environment & ~music).sum()))
print("Speech + Music only:", int((speech & music & ~environment).sum()))
print("Environment + Music only:", int((environment & music & ~speech).sum()))
print("Speech + Environment + Music:", int((speech & environment & music).sum()))

print("\nNo label:")
print("None of them:", int((~speech & ~environment & ~music).sum()))


TOI columns:
['Run_ID', 'Video_ID', 'Start(mins)', 'Start(Secs)', 'Actual_End(mins)', 'Our_End(mins)', 'Our_End(secs)', 'Duration(secs)', 'Video_duration(mins)', 'Video_duration(secs)', 'Start_Stream']

Audio label columns:
['ChunkNumber', 'AudioNumber', 'Speech', 'Environment', 'Music']


,Run_ID,Video_ID,Start(mins),Start(Secs),Actual_End(mins),Our_End(mins),Our_End(secs),Duration(secs),Video_duration(mins),Video_duration(secs),Start_Stream
0,1,video1,00:20:00,20,04:24:00,03:52:00,232,212,15:21:00,921.0,20
1,1,video2,04:44:00,284,08:25:00,08:24:00,504,220,NaN,NaN,284
2,1,video3,08:46:00,526,11:53:00,11:52:00,712,186,NaN,NaN,526
3,1,video4,12:15:00,735,13:17:00,13:17:00,797,62,NaN,NaN,735
4,1,video5,13:38:00,818,15:00:00,15:00:00,900,82,NaN,NaN,818


,ChunkNumber,AudioNumber,Speech,Environment,Music
0,1,1,0,1,0
1,2,1,0,1,0
2,3,1,0,1,0
3,4,1,0,1,0
4,5,1,0,1,0


KeyError: "Cannot find a video/run column in TOI. Columns: ['Run_ID', 'Video_ID', 'Start(mins)', 'Start(Secs)', 'Actual_End(mins)', 'Our_End(mins)', 'Our_End(secs)', 'Duration(secs)', 'Video_duration(mins)', 'Video_duration(secs)', 'Start_Stream']"

In [ ]:
# ============================================================
# 7. Make condition masks
# ============================================================


def make_condition_masks(event_df, conditions):
    masks = {}
    speech = event_df["Speech"].to_numpy().astype(bool)
    env = event_df["Environment"].to_numpy().astype(bool)
    music = event_df["Music"].to_numpy().astype(bool)

    for condition in conditions:
        c = condition.lower()
        if c == "all":
            mask = np.ones(len(event_df), dtype=bool)
        elif c == "speech_only":
            mask = speech & (~env) & (~music)
        elif c == "environment_only":
            mask = env & (~speech) & (~music)
        elif c == "music_only":
            mask = music & (~speech) & (~env)
        elif c == "speech":
            mask = speech
        elif c == "environment":
            mask = env
        elif c == "music":
            mask = music
        else:
            raise ValueError(f"Unknown condition: {condition}")
        masks[condition] = mask
    return masks


condition_masks = make_condition_masks(event_df, CONDITIONS_TO_RUN)

for name, mask in condition_masks.items():
    print(name, ":", int(mask.sum()), "events")

In [ ]:
# ============================================================
# 8. Estimate beta-like responses using delayed-window mean
#    DELAY_SEC = 5, no fMRI normalization
# ============================================================


def estimate_betas_delayed_window_mean_no_fmri_norm(data, event_df, tr=1.0, delay_sec=5.0, window_sec=3.0):
    """
    Creates one beta-like value per event and vertex by averaging fMRI data
    from onset + delay_sec to onset + delay_sec + window_sec.

    No z-scoring/scaling of fMRI is performed.
    """
    data = np.asarray(data, dtype=np.float32)
    n_timepoints, n_vertices = data.shape
    n_events = len(event_df)

    betas = np.full((n_events, n_vertices), np.nan, dtype=np.float32)

    onsets = event_df["onset_sec"].to_numpy(dtype=float)
    delay_tp = int(np.round(delay_sec / tr))
    window_tp = max(1, int(np.round(window_sec / tr)))

    for i, onset in enumerate(onsets):
        start_tp = int(np.round(onset / tr)) + delay_tp
        end_tp = start_tp + window_tp

        if start_tp < 0 or start_tp >= n_timepoints:
            continue

        end_tp = min(end_tp, n_timepoints)
        betas[i, :] = np.nanmean(data[start_tp:end_tp, :], axis=0)

    # Replace non-finite values with zero to keep ridge stable.
    # This is not normalization.
    nonfinite = ~np.isfinite(betas)
    print("Non-finite beta values before replacement:", int(nonfinite.sum()))
    if nonfinite.any():
        betas[nonfinite] = 0.0

    return betas.astype(np.float32)


event_betas = estimate_betas_delayed_window_mean_no_fmri_norm(
    data=data,
    event_df=event_df,
    tr=TR,
    delay_sec=DELAY_SEC,
    window_sec=WINDOW_SEC,
)

print("event_betas shape:", event_betas.shape)
print("Expected: events x vertices")
print("No fMRI normalization was applied.")

In [ ]:
# ============================================================
# 9. Check beta uniqueness
# ============================================================

print("event_betas shape:", event_betas.shape)

for e in [0, 10, 50, 100]:
    if e < event_betas.shape[0]:
        x = event_betas[e, :]
        x = x[np.isfinite(x)]
        unique_count = len(np.unique(np.round(x, 6)))
        print("event", e, "unique beta values:", unique_count, "out of", x.size)

In [ ]:
# ============================================================
# 10. Load model features and align with events
# ============================================================

model_features, loaded_model_name = load_model_features(MODEL_FEATURE_FILE, model_var_name=MODEL_VAR_NAME)
model_features = np.asarray(model_features, dtype=np.float32)

if model_features.ndim != 2:
    raise ValueError(f"model_features must be 2D. Got shape {model_features.shape}")

print("Loaded model variable:", loaded_model_name)
print("Original model_features shape:", model_features.shape)
print("event_betas shape:", event_betas.shape)
print("event_df length:", len(event_df))

# Align by minimum length if needed.
n = min(model_features.shape[0], event_betas.shape[0], len(event_df))
if n != model_features.shape[0] or n != event_betas.shape[0] or n != len(event_df):
    print("WARNING: Length mismatch. Trimming all event-wise matrices to:", n)

model_features = model_features[:n, :]
event_betas = event_betas[:n, :]
event_df = event_df.iloc[:n].reset_index(drop=True)
condition_masks = make_condition_masks(event_df, CONDITIONS_TO_RUN)

print("Aligned model_features shape:", model_features.shape)
print("Aligned event_betas shape:", event_betas.shape)
print("Aligned event_df length:", len(event_df))

for name, mask in condition_masks.items():
    print(name, ":", int(mask.sum()), "events")

In [ ]:
# ============================================================
# 11. Model normalization: demean only
# ============================================================


def demean_model_features_train_test(X_train, X_test):
    """
    Demean model features using the training fold mean only.
    No division by standard deviation.
    """
    mean = np.nanmean(X_train, axis=0, keepdims=True)
    mean[~np.isfinite(mean)] = 0.0

    X_train_dm = X_train - mean
    X_test_dm = X_test - mean

    X_train_dm = np.nan_to_num(X_train_dm, nan=0.0, posinf=0.0, neginf=0.0)
    X_test_dm = np.nan_to_num(X_test_dm, nan=0.0, posinf=0.0, neginf=0.0)

    return X_train_dm.astype(np.float32), X_test_dm.astype(np.float32)

In [ ]:
# ============================================================
# 12. Vertex-wise ridge encoding
#     Model: demean only
#     fMRI/betas: no normalization
# ============================================================


def safe_corr_vectorized(Y_true, Y_pred):
    """
    Fast Pearson correlation for many vertices.
    Inputs: events x vertices.
    Output: vertices.
    """
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    r = np.full(Y_true.shape[1], np.nan, dtype=np.float32)

    for v in range(Y_true.shape[1]):
        yt = Y_true[:, v]
        yp = Y_pred[:, v]
        good = np.isfinite(yt) & np.isfinite(yp)
        if good.sum() < 5:
            continue
        if np.std(yt[good]) < 1e-12 or np.std(yp[good]) < 1e-12:
            continue
        r[v] = pearsonr(yt[good], yp[good])[0]

    return r


def r2_vectorized(Y_true, Y_pred):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    ss_res = np.nansum((Y_true - Y_pred) ** 2, axis=0)
    mean_true = np.nanmean(Y_true, axis=0, keepdims=True)
    ss_tot = np.nansum((Y_true - mean_true) ** 2, axis=0)

    r2 = np.full(Y_true.shape[1], np.nan, dtype=np.float32)
    good = ss_tot > 1e-12
    r2[good] = (1.0 - ss_res[good] / ss_tot[good]).astype(np.float32)
    return r2


def run_vertexwise_ridge_encoding_dmean_no_fmri_norm(
    model_features,
    event_betas,
    valid_events=None,
    alpha=100.0,
    n_splits=5,
    random_seed=42,
):
    """
    Ridge encoding model.

    X/model features: demean only within each training fold.
    Y/fMRI beta values: no normalization, no z-score, no demeaning.
    """

    X = np.asarray(model_features, dtype=np.float32)
    Y = np.asarray(event_betas, dtype=np.float32)

    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"Event mismatch: X has {X.shape[0]}, Y has {Y.shape[0]}")

    if valid_events is None:
        valid_events = np.ones(X.shape[0], dtype=bool)
    else:
        valid_events = np.asarray(valid_events).astype(bool)

    X = X[valid_events]
    Y = Y[valid_events]

    print("Ridge X shape:", X.shape)
    print("Ridge Y shape:", Y.shape)
    print("Model normalization: demean only")
    print("fMRI/beta normalization: none")

    if X.shape[0] < n_splits:
        raise ValueError(f"Too few events for {n_splits}-fold CV: {X.shape[0]}")

    Y = np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    y_pred_all = np.full_like(Y, np.nan, dtype=np.float32)

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=random_seed)

    for fold, (train_idx, test_idx) in enumerate(kfold.split(X), start=1):
        print(f"Fold {fold}/{n_splits}")

        X_train = X[train_idx]
        X_test = X[test_idx]
        Y_train = Y[train_idx]

        # Model features: demean only, using training fold mean.
        X_train_dm, X_test_dm = demean_model_features_train_test(X_train, X_test)

        # fMRI/betas: no normalization.
        # Ridge learns intercept from raw beta scale.
        model = Ridge(alpha=alpha, fit_intercept=True)
        model.fit(X_train_dm, Y_train)
        y_pred_all[test_idx, :] = model.predict(X_test_dm).astype(np.float32)

    vertex_r = safe_corr_vectorized(Y, y_pred_all)
    vertex_r2 = r2_vectorized(Y, y_pred_all)

    print("Finished ridge encoding.")
    print("Finite R:", int(np.isfinite(vertex_r).sum()), "/", vertex_r.size)
    print("Finite R²:", int(np.isfinite(vertex_r2).sum()), "/", vertex_r2.size)
    print("R min/max:", np.nanmin(vertex_r), np.nanmax(vertex_r))
    print("R² min/max:", np.nanmin(vertex_r2), np.nanmax(vertex_r2))

    return vertex_r.astype(np.float32), vertex_r2.astype(np.float32), y_pred_all

In [ ]:
# ============================================================
# 13. Run ridge encoding for each condition
#     Conditions with too few events are skipped safely.
# ============================================================

ridge_results = {}

MIN_EVENTS_REQUIRED = max(2, N_SPLITS)

for condition_name, mask in condition_masks.items():
    n_valid = int(mask.sum())

    print("\n============================================================")
    print("Running condition:", condition_name)
    print("Valid events:", n_valid)
    print("============================================================")

    if n_valid < MIN_EVENTS_REQUIRED:
        print(
            f"Skipping {condition_name}: only {n_valid} events, "
            f"but {MIN_EVENTS_REQUIRED} are required for {N_SPLITS}-fold CV."
        )
        continue

    vertex_r, vertex_r2, y_pred = run_vertexwise_ridge_encoding_dmean_no_fmri_norm(
        model_features=model_features,
        event_betas=event_betas,
        valid_events=mask,
        alpha=RIDGE_ALPHA,
        n_splits=N_SPLITS,
        random_seed=RANDOM_SEED,
    )

    ridge_results[condition_name] = {
        "R": vertex_r,
        "R2": vertex_r2,
    }

    print("R map shape:", vertex_r.shape)
    print("R² map shape:", vertex_r2.shape)
    print("Finite R:", np.sum(np.isfinite(vertex_r)), "/", vertex_r.size)

if len(ridge_results) == 0:
    raise ValueError(
        "No ridge results were created. All selected conditions had too few events. "
        "Try CONDITIONS_TO_RUN = ['all'] or use broader conditions such as 'speech', 'environment', 'music'."
    )


In [ ]:
# ============================================================
# 14. Save Workbench-readable .func.gii maps
# ============================================================


def save_func_gii(out_path, data_1d, hemi):
    """
    Save one vector as Workbench-readable .func.gii.
    """
    x = np.asarray(data_1d, dtype=np.float32).squeeze()

    if x.ndim != 1:
        raise ValueError(f"func.gii data must be 1D. Got shape {x.shape}")

    finite_count = int(np.isfinite(x).sum())
    print("Saving:", out_path)
    print("Map length:", x.size)
    print("Finite values:", finite_count, "/", x.size)

    if finite_count == 0:
        raise ValueError("Map contains no finite values.")

    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    hemi_upper = hemi.upper()
    if hemi_upper == "RIGHT":
        structure = "CortexRight"
    elif hemi_upper == "LEFT":
        structure = "CortexLeft"
    else:
        raise ValueError("HEMI must be RIGHT or LEFT")

    da = nib.gifti.GiftiDataArray(
        data=x,
        intent="NIFTI_INTENT_NORMAL",
        datatype="NIFTI_TYPE_FLOAT32"
    )
    da.meta = nib.gifti.GiftiMetaData({"AnatomicalStructurePrimary": structure})

    img = nib.gifti.GiftiImage(darrays=[da])
    img.meta = nib.gifti.GiftiMetaData({"AnatomicalStructurePrimary": structure})

    nib.save(img, out_path)


os.makedirs(OUT_DIR, exist_ok=True)
print("Saving maps to:", OUT_DIR)

for condition_name, res in ridge_results.items():
    R_map = res["R"]
    R2_map = res["R2"]

    R_out = os.path.join(
        OUT_DIR,
        f"{MODEL_NAME}_RIDGE_dmean_delay5_noFMRIz_R_{condition_name}_{HEMI}.func.gii"
    )

    R2_out = os.path.join(
        OUT_DIR,
        f"{MODEL_NAME}_RIDGE_dmean_delay5_noFMRIz_R2_{condition_name}_{HEMI}.func.gii"
    )

    save_func_gii(R_out, R_map, HEMI)
    save_func_gii(R2_out, R2_map, HEMI)

print("Done saving.")

In [ ]:
# ============================================================
# 15. Check saved maps
# ============================================================

for condition_name in ridge_results.keys():
    R_out = os.path.join(
        OUT_DIR,
        f"{MODEL_NAME}_RIDGE_dmean_delay5_noFMRIz_R_{condition_name}_{HEMI}.func.gii"
    )

    img = nib.load(R_out)
    x = img.darrays[0].data

    print("
File:", R_out)
    print("Shape:", x.shape)
    print("dtype:", x.dtype)
    print("finite:", int(np.isfinite(x).sum()), "/", x.size)
    print("min/max:", np.nanmin(x), np.nanmax(x))
    print("unique rounded values:", len(np.unique(np.round(x[np.isfinite(x)], 6))))

## Workbench loading notes

Open the matching surface first:

- RIGHT maps → right hemisphere fsLR 32k surface, for example `S1200.R.flat.32k_fs_LR.surf.gii` or right midthickness.
- LEFT maps → left hemisphere fsLR 32k surface.

Then load the `.func.gii` output files as overlay maps.

If the output still looks like parcels, check the unique-value diagnostic cells. Ridge regression cannot create real vertex-wise detail if the input fMRI matrix is already parcellated or parcel-expanded.